In [31]:
import pandas as pd
import os
import sys


base_dir = "/Users/jinzhou/Applications/d3js_vine"
logs_dir = os.path.join(base_dir, 'logs')
with os.scandir(logs_dir) as entries:
    for entry in entries:
        if entry.is_dir():
            log_dir = os.path.join('logs', entry.name)
            break
dirname = os.path.join(log_dir, 'vine-logs')
task_dag_csv = os.path.join(dirname, 'task_dag.csv')

In [32]:
task_done_df = pd.read_csv(os.path.join(dirname, 'task_done.csv'))
task_done_df = task_done_df[task_done_df['is_recovery_task'] == False]
task_dag_df = task_done_df[['task_id', 'category', 'input_files', 'output_files']]
# save
task_dag_df.to_csv(task_dag_csv, index=False)

In [33]:
class EdgeNode:
    def __init__(self, tail, head, tail_link=None, head_link=None):
        self.tail = tail
        self.head = head
        self.tail_link = tail_link
        self.head_link = head_link

class VertexNode:
    def __init__(self, vertex_id):
        self.vertex_id = vertex_id
        self.first_in = None
        self.first_out = None

class OrthogonalListGraph:
    def __init__(self):
        self.vertices = {}
        self.edges = set()

    def add_vertex(self, vertex_id):
        if vertex_id not in self.vertices:
            self.vertices[vertex_id] = VertexNode(vertex_id)

    def add_edge(self, tail, head):
        if (tail, head) in self.edges:
            return
        new_edge = EdgeNode(tail, head)
        
        if tail in self.vertices:
            new_edge.tail_link = self.vertices[tail].first_out
            self.vertices[tail].first_out = new_edge
        if head in self.vertices:
            new_edge.head_link = self.vertices[head].first_in
            self.vertices[head].first_in = new_edge

        self.edges.add((tail, head))

    def display(self):
        for v in self.vertices.values():
            print(f"Vertex {v.vertex_id}:")
            out_edge = v.first_out
            while out_edge:
                print(f"  Out to {out_edge.head}")
                out_edge = out_edge.tail_link
            in_edge = v.first_in
            while in_edge:
                print(f"  In from {in_edge.tail}")
                in_edge = in_edge.head_link

graph = OrthogonalListGraph()

In [34]:
import json

def print_json(json_obj):
    print(json.dumps(json_obj, indent=2))

In [35]:
import ast

input_to_tasks = {}


for index, row in task_dag_df.iterrows():
    graph.add_vertex(row['task_id'])
    input_files = ast.literal_eval(row['input_files'])
    for file in input_files:
        if file not in input_to_tasks:
            input_to_tasks[file] = []
        input_to_tasks[file].append(row['task_id'])


for index, row in task_dag_df.iterrows():
    output_files = ast.literal_eval(row['output_files'])
    for file in output_files:
        if file in input_to_tasks:
            for target_task_id in input_to_tasks[file]:
                graph.add_edge(row['task_id'], target_task_id)

graph.display()


Vertex 1:
  Out to 3
  Out to 2
Vertex 2:
  Out to 7
  Out to 6
  In from 1
Vertex 3:
  Out to 5
  Out to 4
  In from 1
Vertex 4:
  Out to 9
  Out to 8
  In from 3
Vertex 5:
  Out to 13
  Out to 12
  In from 3
Vertex 6:
  Out to 15
  Out to 14
  In from 2
Vertex 7:
  Out to 11
  Out to 10
  In from 2
Vertex 8:
  Out to 27
  Out to 26
  In from 4
Vertex 9:
  Out to 17
  Out to 16
  In from 4
Vertex 10:
  Out to 19
  Out to 18
  In from 7
Vertex 11:
  Out to 25
  Out to 24
  In from 7
Vertex 12:
  Out to 23
  Out to 22
  In from 5
Vertex 13:
  Out to 41
  Out to 40
  In from 5
Vertex 14:
  Out to 21
  Out to 20
  In from 6
Vertex 15:
  Out to 39
  Out to 38
  In from 6
Vertex 16:
  Out to 29
  Out to 28
  In from 9
Vertex 17:
  Out to 61
  Out to 60
  In from 9
Vertex 18:
  Out to 63
  Out to 62
  In from 10
Vertex 19:
  Out to 31
  Out to 30
  In from 10
Vertex 20:
  Out to 67
  Out to 66
  In from 14
Vertex 21:
  Out to 75
  Out to 74
  In from 14
Vertex 22:
  Out to 65
  Out to 64
  I

In [36]:
import graphviz

def visualize_graph(graph):
    dot = graphviz.Digraph(comment='Task Graph')
    
    for vertex_id, vertex in graph.vertices.items():
        dot.node(str(vertex_id), str(vertex_id))
        
    for vertex_id, vertex in graph.vertices.items():
        edge = vertex.first_out
        while edge:
            dot.edge(str(edge.tail), str(edge.head))
            edge = edge.tail_link

    # 保存和渲染图形
    dot.render('output/task_graph', format='png', view=True)

visualize_graph(graph)